In [ ]:
from tbparse import SummaryReader
import pandas as pd
import plotly.graph_objs as go
import plotly.express as px
import numpy as np
from plotly.colors import hex_to_rgb
from pathlib import Path

pd.options.plotting.backend = "plotly"

In [ ]:


MODULE_COLS = ["nereus_modules.social", "nereus_modules.map", "nereus_modules.prior"]
MODULE_TITLES = {
    "nereus_modules.social": "Social",
    "nereus_modules.map": "Map",
    "nereus_modules.prior": "Prior",
}
PER_STEP_COLS = ["fde", "k_fde"]
SCALAR_COLS = ["ade", "k_ade", "pred_risk", "min_pred_dist", "collision_ratio", "n_graphs"]
# STEP_SIZE = 10 in utils.config -> 6 prediction steps per minute.
STEPS_PER_MINUTE = 6
PAPER_COLORS = px.colors.qualitative.Set2
SHIP_GROUPS = ["all", "cargo", "passenger", "sailing", "other"]


In [ ]:
def load_hp(reader, columns=None):
    df_hp = reader.hparams
    df_hp = df_hp.drop_duplicates(subset=["dir_name", "tag"])
    df_hp = df_hp.pivot(index="dir_name", columns="tag", values="value")
    df_hp["v_name"] = df_hp.index.str.split("/").str[-1]

    if columns is None:
        return df_hp
    return df_hp[columns]

def load_step_metrics(reader, metrics=None):
    df_metrics = reader.scalars
    df_metrics = df_metrics[df_metrics["tag"].str.contains("step")]
    df_metrics = df_metrics.drop_duplicates(subset=["dir_name", "tag", "step"])
    df_metrics = df_metrics.pivot(index="step", columns=["tag", "dir_name"], values="value")

    if metrics is None:
        return df_metrics
    return df_metrics[metrics]

def load_metrics(reader, metrics=None):
    df_metrics = reader.scalars
    df_metrics = df_metrics[(~df_metrics["tag"].str.contains("step")) & (df_metrics["tag"] != "epoch") & (df_metrics["tag"] != "hp_metric")]
    df_metrics = df_metrics.drop_duplicates(subset=["dir_name", "tag", "step"])
    df_metrics = df_metrics.pivot(index="step", columns=["tag", "dir_name"], values="value")

    if metrics is None:
        return df_metrics
    return df_metrics[metrics]




def _rgba(c, a):
    """Plotly color string ('#rrggbb' or 'rgb(r,g,b)') -> 'rgba(r,g,b,a)'."""
    if c.startswith("#"):
        r, g, b = hex_to_rgb(c)
    else:
        r, g, b = (int(x) for x in c[c.find("(") + 1:c.find(")")].split(",")[:3])
    return f"rgba({r},{g},{b},{a})"


def _setting_label(series):
    """Map a module hyperparameter column to clean on/off-style labels ('off' when disabled)."""
    s = series.where(series.notna(), "off")
    return s.replace({None: "off", "null": "off", "None": "off"}).astype(str)


def build_setting_colors(df_hp, modules=MODULE_COLS, palette=PAPER_COLORS):
    """Fixed color per setting value across every module/plot (e.g. 'off' is always one color)."""
    labels = sorted(pd.unique(pd.concat([_setting_label(df_hp[m]) for m in modules])))
    return {lab: palette[i % len(palette)] for i, lab in enumerate(labels)}


# Global lookup so the same setting value always gets the same color.


def _setting_color(setting):
    """Color for a setting, extending SETTING_COLORS deterministically for unseen values."""
    if setting not in SETTING_COLORS:
        SETTING_COLORS[setting] = PAPER_COLORS[len(SETTING_COLORS) % len(PAPER_COLORS)]
    return SETTING_COLORS[setting]


def plot_module_effect(long, df_hp, module, metric="fde", region="kiel", ship_group="all",
                       agg="mean", show_bands=True):
    """Paper figure: per-step `metric` vs prediction horizon, one line per setting of `module`.

    Runs are pooled by the chosen module's value (the other modules vary underneath),
    so each line is the mean over runs with that setting; when `show_bands` is True the
    shaded band is ±1 std. Colors are taken from the global SETTING_COLORS so the same
    value is drawn identically across all figures.

    Works with either `load_eval_csvs` output or the old `df_step_metrics` once passed
    through `step_metrics_to_long`. Pass `region=None` for the old logs (no region column).
    """
    df = long[long["ship_group"] == ship_group].copy()
    if region is not None:
        df = df[df["region"] == region]
    df["setting"] = _setting_label(df["dir_name"].map(df_hp[module]))

    stats = (df.groupby(["setting", "minute"])[metric]
               .agg(mean=agg, std="std").reset_index().sort_values("minute"))

    fig = go.Figure()
    for setting, sdf in stats.groupby("setting"):
        color = _setting_color(setting)
        x, y, e = sdf["minute"].to_numpy(), sdf["mean"].to_numpy(), sdf["std"].fillna(0).to_numpy()
        if show_bands:
            fig.add_trace(go.Scatter(
                x=np.concatenate([x, x[::-1]]), y=np.concatenate([y + e, (y - e)[::-1]]),
                fill="toself", fillcolor=_rgba(color, 0.15), line=dict(width=0),
                hoverinfo="skip", showlegend=False, legendgroup=setting))
        fig.add_trace(go.Scatter(
            x=x, y=y, mode="lines", name=setting, legendgroup=setting,
            line=dict(color=color, width=2.5)))

    n = df["dir_name"].nunique()
    fig.update_layout(
        template="simple_white", width=560, height=420, font=dict(size=15),
        title=f"{MODULE_TITLES[module]} module &nbsp;({n} runs, {region or 'all regions'})",
        xaxis_title="Prediction horizon [min]", yaxis_title=f"{metric.upper()} [m]",
        legend_title_text=MODULE_TITLES[module], margin=dict(t=55, b=55, l=65, r=20))
    return fig



class HyperparamMetricAnalyzer:
    """
    Analyze and visualize time series metrics grouped by hyperparameters.

    Args:
        hyperparams_df: DataFrame with hyperparameters (index=key)
        metrics_df: DataFrame with MultiIndex columns (metric_name, key) and time series as rows
    """

    def __init__(self, hyperparams_df, metrics_df):
        self.hyperparams = hyperparams_df
        self.metrics = metrics_df
        self._validate_data()

    def _validate_data(self):
        """Check that keys match between dataframes."""
        hp_keys = set(self.hyperparams.index)
        metric_keys = set(self.metrics.columns.get_level_values(1))

        if not hp_keys == metric_keys:
            missing_in_metrics = hp_keys - metric_keys
            missing_in_hp = metric_keys - hp_keys
            if missing_in_metrics:
                print(f"Warning: Keys in hyperparams but not metrics: {missing_in_metrics}")
            if missing_in_hp:
                print(f"Warning: Keys in metrics but not hyperparams: {missing_in_hp}")

    def _filter_keys(self, hyperparam_filter=None):
        """
        Filter keys based on hyperparameter values.

        Args:
            hyperparam_filter: Dict with hyperparam names as keys and list of allowed values as values
                              e.g., {'learning_rate': [0.001, 0.01], 'optimizer': ['adam']}

        Returns:
            List of filtered keys
        """
        if hyperparam_filter is None:
            return self.hyperparams.index.tolist()

        mask = pd.Series(True, index=self.hyperparams.index)
        for hp_name, allowed_values in hyperparam_filter.items():
            if hp_name not in self.hyperparams.columns:
                raise ValueError(
                    f"Hyperparameter '{hp_name}' not found. Available: {self.hyperparams.columns.tolist()}"
                )
            mask &= self.hyperparams[hp_name].isin(allowed_values)

        return self.hyperparams[mask].index.tolist()

    def get_metric(self, metric_name):
        """
        Extract a single metric as DataFrame with keys as columns and forecast horizon as rows.

        Returns:
            DataFrame with time series for each key
        """
        if metric_name not in self.metrics.columns.get_level_values(0):
            raise ValueError(f"Metric '{metric_name}' not found. Available: {self.get_metric_names()}")
        return self.metrics[metric_name]

    def get_metric_names(self):
        """Return list of available metric names."""
        return self.metrics.columns.get_level_values(0).unique().tolist()

    def plot_metric_timeseries(
        self,
        metric_name,
        hyperparam=None,
        hyperparam_values=None,
        hyperparam_filter=None,
        height=600,
        width=1000,
        show_bands=False,
    ):
        """
        Plot time series of a metric, optionally grouped by hyperparameter(s).

        Args:
            metric_name: Name of the metric to plot
            hyperparam: Optional hyperparameter(s) to color/group by. Can be:
                       - Single string: 'learning_rate'
                       - List of strings: ['learning_rate', 'optimizer']
            hyperparam_values: Optional list of specific values for the grouping hyperparam to plot
                              Only works with single hyperparam grouping
            hyperparam_filter: Optional dict to filter by other hyperparameters
                              e.g., {'learning_rate': [0.001, 0.01], 'optimizer': ['adam']}
            height: Figure height in pixels
            width: Figure width in pixels
            show_bands: If True, show mean ± std bands when grouping by hyperparam

        Returns:
            Plotly figure
        """
        metric_df = self.get_metric(metric_name)

        # Apply hyperparameter filter
        filtered_keys = self._filter_keys(hyperparam_filter)
        metric_df = metric_df[[k for k in filtered_keys if k in metric_df.columns]]

        fig = go.Figure()

        if hyperparam is None:
            # Plot all time series
            for key in metric_df.columns:
                fig.add_trace(go.Scatter(x=metric_df.index, y=metric_df[key], mode="lines", name=key, opacity=0.7))
            title = f"{metric_name} Time Series - All Runs"
        else:
            # Handle single or multiple hyperparameters
            if isinstance(hyperparam, str):
                hyperparam = [hyperparam]

            # Validate hyperparameters
            for hp in hyperparam:
                if hp not in self.hyperparams.columns:
                    raise ValueError(f"Hyperparameter '{hp}' not found. Available: {self.hyperparams.columns.tolist()}")

            # Create group labels by combining hyperparameter values
            group_labels = {}
            for key in metric_df.columns:
                if key in self.hyperparams.index:
                    label_parts = [f"{hp}={self.hyperparams.loc[key, hp]}" for hp in hyperparam]
                    group_labels[key] = ", ".join(label_parts)

            # Further filter by specific values if provided (only for single hyperparam)
            if hyperparam_values is not None:
                if len(hyperparam) > 1:
                    print("Warning: hyperparam_values only works with single hyperparam grouping. Ignoring.")
                else:
                    keys_to_plot = [
                        k
                        for k in metric_df.columns
                        if k in self.hyperparams.index and self.hyperparams.loc[k, hyperparam[0]] in hyperparam_values
                    ]
                    group_labels = {k: v for k, v in group_labels.items() if k in keys_to_plot}

            # Group data by combined labels
            grouped_data = {}
            for key, label in group_labels.items():
                if label not in grouped_data:
                    grouped_data[label] = []
                grouped_data[label].append(metric_df[key].values)

            # Plot grouped time series
            color_sequence = px.colors.qualitative.Plotly  # Use Plotly's default color sequence
            for color_idx, (group_label, series_list) in enumerate(grouped_data.items()):
                series_array = np.array(series_list)
                mean_series = np.nanmean(series_array, axis=0)
                color = color_sequence[color_idx % len(color_sequence)]

                if show_bands and len(series_list) > 1:
                    std_series = np.nanstd(series_array, axis=0)

                    # Get opaque version of color for band
                    band_color = get_opaque_version(color, opacity=0.2)

                    # Handle NaN values - only plot where we have valid data
                    valid_mask = ~np.isnan(mean_series)
                    if valid_mask.any():
                        valid_indices = metric_df.index[valid_mask]
                        valid_mean = mean_series[valid_mask]
                        valid_std = std_series[valid_mask]

                        # Add confidence band
                        fig.add_trace(
                            go.Scatter(
                                x=np.concatenate([valid_indices, valid_indices[::-1]]),
                                y=np.concatenate([valid_mean + valid_std, (valid_mean - valid_std)[::-1]]),
                                fill="toself",
                                fillcolor=band_color,
                                line=dict(color="rgba(255,255,255,0)"),
                                showlegend=False,
                                name=f"{group_label} band",
                                legendgroup=group_label,
                            )
                        )

                # Add mean line (handle NaN values)
                fig.add_trace(
                    go.Scatter(
                        x=metric_df.index,
                        y=mean_series,
                        mode="lines",
                        name=group_label,
                        line=dict(width=2, color=color),
                        legendgroup=group_label,
                    )
                )

                # Optionally add individual runs with low opacity and matching color
                if not show_bands:
                    for i, series in enumerate(series_list):
                        fig.add_trace(
                            go.Scatter(
                                x=metric_df.index,
                                y=series,
                                mode="lines",
                                name=group_label,
                                line=dict(width=1, color=color),
                                opacity=0.3,
                                showlegend=False,#(i == 0),
                                legendgroup=group_label,
                            )
                        )

            hyperparam_str = " & ".join(hyperparam)
            title = f"{metric_name} by {hyperparam_str}"

        fig.update_layout(
            title=title,
            xaxis_title="Forecast Horizon",
            yaxis_title=metric_name,
            height=height,
            width=width,
            hovermode="x unified",
        )

        return fig

    def plot_metric_comparison(
        self, metric_names, hyperparam, hyperparam_values=None, hyperparam_filter=None, height=600, width=1200
    ):
        """
        Compare multiple metrics side-by-side across forecast horizon.

        Args:
            metric_names: List of metric names to compare
            hyperparam: Hyperparameter(s) to group by. Can be:
                       - Single string: 'learning_rate'
                       - List of strings: ['learning_rate', 'optimizer']
            hyperparam_values: Optional list of specific values for the grouping hyperparam to plot
                              Only works with single hyperparam grouping
            hyperparam_filter: Optional dict to filter by other hyperparameters
                              e.g., {'learning_rate': [0.001, 0.01], 'batch_size': [32]}
            height: Figure height in pixels
            width: Figure width in pixels
        """
        fig = make_subplots(rows=1, cols=len(metric_names), subplot_titles=metric_names, shared_xaxes=True)

        # Handle single or multiple hyperparameters
        if isinstance(hyperparam, str):
            hyperparam = [hyperparam]

        # Validate hyperparameters
        for hp in hyperparam:
            if hp not in self.hyperparams.columns:
                raise ValueError(f"Hyperparameter '{hp}' not found. Available: {self.hyperparams.columns.tolist()}")

        # Apply hyperparameter filter
        filtered_keys = self._filter_keys(hyperparam_filter)

        # Create group labels by combining hyperparameter values
        group_labels = {}
        for key in filtered_keys:
            if key in self.hyperparams.index:
                label_parts = [f"{hp}={self.hyperparams.loc[key, hp]}" for hp in hyperparam]
                group_labels[key] = ", ".join(label_parts)

        # Further filter by specific values if provided (only for single hyperparam)
        if hyperparam_values is not None:
            if len(hyperparam) > 1:
                print("Warning: hyperparam_values only works with single hyperparam grouping. Ignoring.")
            else:
                keys_to_plot = [
                    k
                    for k in filtered_keys
                    if k in self.hyperparams.index and self.hyperparams.loc[k, hyperparam[0]] in hyperparam_values
                ]
                group_labels = {k: v for k, v in group_labels.items() if k in keys_to_plot}

        for col_idx, metric_name in enumerate(metric_names, 1):
            metric_df = self.get_metric(metric_name)

            # Group data by combined labels
            grouped_data = {}
            for key, label in group_labels.items():
                if key in metric_df.columns:
                    if label not in grouped_data:
                        grouped_data[label] = []
                    grouped_data[label].append(metric_df[key].values)

            # Plot each group
            for group_label, series_list in grouped_data.items():
                mean_series = np.mean(series_list, axis=0)

                fig.add_trace(
                    go.Scatter(
                        x=metric_df.index,
                        y=mean_series,
                        mode="lines",
                        name=group_label,
                        showlegend=(col_idx == 1),
                        legendgroup=group_label,
                    ),
                    row=1,
                    col=col_idx,
                )

            fig.update_xaxes(title_text="Forecast Horizon", row=1, col=col_idx)
            fig.update_yaxes(title_text=metric_name, row=1, col=col_idx)

        hyperparam_str = " & ".join(hyperparam)
        fig.update_layout(
            title_text=f"Metrics Comparison by {hyperparam_str}", height=height, width=width, hovermode="x unified"
        )

        return fig

    def aggregate_timeseries(self, metric_name, hyperparam, agg_func="mean", hyperparam_filter=None):
        """
        Aggregate time series grouped by a hyperparameter.

        Args:
            metric_name: Name of the metric to aggregate
            hyperparam: Hyperparameter column name to group by
            agg_func: Aggregation function ('mean', 'median', 'min', 'max', 'std')
            hyperparam_filter: Optional dict to filter by other hyperparameters

        Returns:
            DataFrame with aggregated time series per hyperparameter value
        """
        metric_df = self.get_metric(metric_name)
        hp_series = self.hyperparams[hyperparam]

        # Apply hyperparameter filter
        filtered_keys = self._filter_keys(hyperparam_filter)

        # Group keys by hyperparameter value
        grouped_data = {}
        for key in filtered_keys:
            if key in metric_df.columns and key in hp_series.index:
                hp_value = hp_series.loc[key]
                if hp_value not in grouped_data:
                    grouped_data[hp_value] = []
                grouped_data[hp_value].append(metric_df[key].values)

        # Aggregate each group
        result = {}
        for hp_value, series_list in grouped_data.items():
            series_array = np.array(series_list)
            if agg_func == "mean":
                result[hp_value] = np.mean(series_array, axis=0)
            elif agg_func == "median":
                result[hp_value] = np.median(series_array, axis=0)
            elif agg_func == "min":
                result[hp_value] = np.min(series_array, axis=0)
            elif agg_func == "max":
                result[hp_value] = np.max(series_array, axis=0)
            elif agg_func == "std":
                result[hp_value] = np.std(series_array, axis=0)

        return pd.DataFrame(result, index=metric_df.index)

    def aggregate_over_horizon(self, metric_name, hyperparam, agg_func="mean", hyperparam_filter=None):
        """
        Aggregate metric values over the entire forecast horizon, grouped by hyperparameter.

        Args:
            metric_name: Name of the metric to aggregate
            hyperparam: Hyperparameter column name to group by
            agg_func: Aggregation function ('mean', 'median', 'min', 'max', 'std')
            hyperparam_filter: Optional dict to filter by other hyperparameters

        Returns:
            Series with aggregated scalar values per hyperparameter value
        """
        metric_df = self.get_metric(metric_name)
        hp_series = self.hyperparams[hyperparam]

        # Apply hyperparameter filter
        filtered_keys = self._filter_keys(hyperparam_filter)

        # First aggregate each time series to a scalar
        scalar_metrics = {}
        for key in filtered_keys:
            if key in metric_df.columns and key in hp_series.index:
                if agg_func == "mean":
                    scalar_metrics[key] = metric_df[key].mean()
                elif agg_func == "median":
                    scalar_metrics[key] = metric_df[key].median()
                elif agg_func == "min":
                    scalar_metrics[key] = metric_df[key].min()
                elif agg_func == "max":
                    scalar_metrics[key] = metric_df[key].max()
                elif agg_func == "std":
                    scalar_metrics[key] = metric_df[key].std()

        # Then group by hyperparameter
        combined = pd.DataFrame({"metric": scalar_metrics, "hyperparam": hp_series})
        return combined.groupby("hyperparam")["metric"].mean().sort_index()

    def plot_aggregated_by_hyperparam(
        self, metric_name, hyperparam, kind="box", hyperparam_filter=None, height=600, width=1000
    ):
        """
        Plot aggregated metric values (over forecast horizon) by hyperparameter.

        Args:
            metric_name: Name of the metric to plot
            hyperparam: Hyperparameter to group by
            kind: Plot type ('box', 'violin', 'bar')
            hyperparam_filter: Optional dict to filter by other hyperparameters
            height: Figure height in pixels
            width: Figure width in pixels
        """
        metric_df = self.get_metric(metric_name)
        hp_series = self.hyperparams[hyperparam]

        # Apply hyperparameter filter
        filtered_keys = self._filter_keys(hyperparam_filter)

        # Aggregate each time series to a scalar (mean over horizon)
        scalar_metrics = {}
        for key in filtered_keys:
            if key in metric_df.columns and key in hp_series.index:
                scalar_metrics[key] = metric_df[key].mean()

        plot_df = pd.DataFrame({"metric": scalar_metrics, "hyperparam": hp_series.astype(str)}).dropna()

        title = f"{metric_name} (avg over horizon) by {hyperparam}"

        if kind == "box":
            fig = px.box(
                plot_df,
                x="hyperparam",
                y="metric",
                title=title,
                labels={"hyperparam": hyperparam, "metric": f"{metric_name} (mean)"},
            )
        elif kind == "violin":
            fig = px.violin(
                plot_df,
                x="hyperparam",
                y="metric",
                title=title,
                box=True,
                points="all",
                labels={"hyperparam": hyperparam, "metric": f"{metric_name} (mean)"},
            )
        elif kind == "bar":
            agg_data = plot_df.groupby("hyperparam")["metric"].agg(["mean", "std"]).reset_index()
            fig = go.Figure()
            fig.add_trace(
                go.Bar(
                    x=agg_data["hyperparam"],
                    y=agg_data["mean"],
                    error_y=dict(type="data", array=agg_data["std"]),
                    name=metric_name,
                )
            )
            fig.update_layout(title=title, xaxis_title=hyperparam, yaxis_title=f"{metric_name} (mean)")

        fig.update_layout(height=height, width=width)
        return fig

    def get_best_configs(self, metric_name, n=5, maximize=True, horizon_agg="mean", hyperparam_filter=None):
        """
        Get top N configurations based on a metric (aggregated over forecast horizon).

        Args:
            metric_name: Metric to rank by
            n: Number of top configs to return
            maximize: If True, return highest values; if False, return lowest
            horizon_agg: How to aggregate over horizon ('mean', 'median', 'last', 'min', 'max')
            hyperparam_filter: Optional dict to filter by hyperparameters before ranking

        Returns:
            DataFrame with top N configs and their hyperparameters
        """
        metric_df = self.get_metric(metric_name)

        # Apply hyperparameter filter
        filtered_keys = self._filter_keys(hyperparam_filter)

        # Aggregate each time series to a scalar
        scalar_metrics = {}
        for key in filtered_keys:
            if key in metric_df.columns:
                if horizon_agg == "mean":
                    scalar_metrics[key] = metric_df[key].mean()
                elif horizon_agg == "median":
                    scalar_metrics[key] = metric_df[key].median()
                elif horizon_agg == "last":
                    scalar_metrics[key] = metric_df[key].iloc[-1]
                elif horizon_agg == "min":
                    scalar_metrics[key] = metric_df[key].min()
                elif horizon_agg == "max":
                    scalar_metrics[key] = metric_df[key].max()

        metric_series = pd.Series(scalar_metrics).sort_values(ascending=not maximize)
        top_keys = metric_series.head(n).index

        result = self.hyperparams.loc[top_keys].copy()
        result[f"{metric_name}_({horizon_agg})"] = metric_series.loc[top_keys]

        return result

In [ ]:
#log_dir = "lightning_logs/nereus"
log_dir = "lightning_logs/nereus_ablation/"

reader = SummaryReader(log_dir,extra_columns={"dir_name"})
df_metrics = reader.scalars
df_loss = load_metrics(reader)
df_hp = load_hp(reader)
df_step_metrics = load_step_metrics(reader)


SETTING_COLORS = build_setting_colors(df_hp)


In [ ]:
pivot = df_metrics[df_metrics["tag"] == "val_metric"].pivot(columns="dir_name", values="value", index="step")
fig = go.Figure()
for col in pivot.columns:
    s = pivot[col].dropna()
    fig.add_trace(go.Scatter(x=s.index, y=s.values, mode="lines", name=col))
fig.show()

In [ ]:
analyzer = HyperparamMetricAnalyzer(df_hp, df_step_metrics)
print("Available metrics:", analyzer.get_metric_names())

In [ ]:

hyperparam = ["nereus_modules.map"]#
hyperparam = ["nereus_modules.prior"]#"nereus_modules.prior"
hyperparam = ["nereus_modules.social"]#"nereus_modules.prior"

fig = analyzer.plot_metric_timeseries('step_de/val', hyperparam=hyperparam, show_bands=False,
                                    hyperparam_filter={
                                            'model': ["NereusModule"],
                                                       })

fig.show()


## New `full_eval` metrics → old format + paper ablation plots

`load_eval_csvs` reads every `version_*/eval_<source>.csv` written by `full_eval_nereus.py`.
`eval_step_metrics` reshapes the per-timestep columns (`fde`, `k_fde`) into the same
`(metric, dir_name) × step` layout as `df_step_metrics`, so it drops straight into
`HyperparamMetricAnalyzer(df_hp, ...)`. `eval_scalar_metrics` returns the horizon-independent
scalars (`ade`, `collision_ratio`, …), one row per model.

In [ ]:



def load_eval_csvs(log_dir, source="fh"):
    """Concatenate every per-model full_eval CSV, tagged by version (`dir_name`)."""
    frames = []
    for csv_path in sorted(Path(log_dir).glob(f"version_*/eval_{source}.csv")):
        df = pd.read_csv(csv_path)
        df["dir_name"] = csv_path.parent.name
        frames.append(df)
    if not frames:
        raise FileNotFoundError(f"No eval_{source}.csv files under {log_dir}")
    return pd.concat(frames, ignore_index=True)


def eval_step_metrics(long, region=None, ship_group="all", metrics=PER_STEP_COLS):
    """Reshape into the `df_step_metrics` format: MultiIndex (metric, dir_name) × step."""
    df = long[long["ship_group"] == ship_group]
    if region is not None:
        df = df[df["region"] == region]
    return df.pivot_table(index="step", columns="dir_name", values=list(metrics), aggfunc="mean")


def eval_scalar_metrics(long, region=None, ship_group="all"):
    """One row per model of the horizon-independent scalars (ade, collision, …)."""
    df = long[long["ship_group"] == ship_group]
    if region is not None:
        df = df[df["region"] == region]
    return df.groupby("dir_name")[SCALAR_COLS].mean()


In [ ]:



def step_metrics_to_long(df_step, split="test", tag_map=None, ship_group="all"):
    """Transform the old `df_step_metrics` into the same long schema as `load_eval_csvs`.

    `df_step_metrics` has `step` as index and MultiIndex `(tag, dir_name)` columns
    (e.g. tag `step_de/test`). This melts it and renames the tbparse tags to the
    full_eval metric names, so the result feeds `plot_module_effect` unchanged:
        step_de/<split>   -> fde
        step_k_de/<split> -> k_fde

    Args:
        split:      logged split to pull ("test" or "val").
        tag_map:    optional {tag: metric} override.
        ship_group: value for the synthesized `ship_group` column (old logs have none).
    """
    tag_map = tag_map or {f"step_de/{split}": "fde", f"step_k_de/{split}": "k_fde"}
    tags = df_step.columns.get_level_values(0)
    frames = []
    for tag, metric in tag_map.items():
        if tag not in tags:
            continue
        long = (df_step[tag].reset_index()
                .melt(id_vars="step", var_name="dir_name", value_name=metric)
                .set_index(["step", "dir_name"]))
        frames.append(long)
    if not frames:
        raise ValueError(f"No tags for split={split!r}; available: {sorted(set(tags))}")
    out = pd.concat(frames, axis=1).reset_index()
    # old series is 0-indexed by timestep; normalize so the first step is 1/STEPS_PER_MINUTE min
    out["minute"] = (out["step"] - out["step"].min() + 1) / STEPS_PER_MINUTE
    out["ship_group"] = ship_group
    return out


In [ ]:
# Load the new metrics and plug them into the existing analyzer / hp table.
eval_long = load_eval_csvs(log_dir, source="fh")
df_eval_step = eval_step_metrics(eval_long, region="kiel", ship_group="all")
df_eval_scalar = eval_scalar_metrics(eval_long, region="kiel", ship_group="all")

print("models evaluated:", eval_long["dir_name"].nunique(), "|", sorted(eval_long["dir_name"].unique()))

# Same format as df_step_metrics -> reuse the analyzer directly.
eval_analyzer = HyperparamMetricAnalyzer(df_hp, df_eval_step)
print("metrics:", eval_analyzer.get_metric_names())

# Scalars joined with the module hyperparameters, ready for tables/plots.
df_eval_scalar = df_eval_scalar.join(df_hp[MODULE_COLS])


In [ ]:
df_eval_scalar.groupby(MODULE_COLS).agg("mean")[["ade","k_ade"]]

In [ ]:
df_eval_scalar

In [ ]:
df_eval_step.loc[[6,12,30],("fde","version_8")]
df_eval_step.loc[[6,12,30],("fde","version_15")]

In [ ]:
df_eval_scalar.pivot(columns="nereus_modules.prior", values="ade",index=["nereus_modules.map","nereus_modules.social"]).T#.sort_values("ade")

In [ ]:
#df_eval_scalar.pivot_table(index='some_id_column', columns='nereus_modules.prior', values='ade', aggfunc='first')


In [ ]:

# ── Helpers ───────────────────────────────────────────────────────────────────
def clean_modules(df):
    out = df.copy()
    for col in MODULE_COLS:
        out[col] = out[col].replace({None: "off", "null": "off", "None": "off", float("nan"): "off"})
    return out


def fde_at_steps(df_step, steps=[6,18,30], metric="fde"):
    frames = {}
    for s in steps:
        frames[f"{metric}@{s}"] = df_step.loc[s, metric]
    return pd.DataFrame(frames)


def build_table(df_scalar, df_step, group_col,fde_steps=[6,18,30],show_std=False):
    fde_df  = fde_at_steps(df_step, steps=fde_steps, metric="fde")
    kfde_df = fde_at_steps(df_step, steps=fde_steps, metric="k_fde")

    scalar = clean_modules(df_scalar)
    combined = (
        scalar[MODULE_COLS + DISPLAY_SCALAR]
        .join(fde_df)
        .join(kfde_df)
    )
    metric_cols = [c for c in combined.columns if c not in MODULE_COLS]

    agg = combined.groupby(group_col)[metric_cols].agg(["mean", "std"]).round(4)

    # Sort so "off" is always first, rest alphabetically
    order = sorted(agg.index, key=lambda x: (x != "off", x))
    agg = agg.loc[order]

    if not show_std:
        return agg.xs("mean", axis=1, level=1)

    rows = {}
    for idx in agg.index:
        rows[idx] = {
            col: f"{agg.loc[idx, (col, 'mean')]:{FLOAT_FMT[2:-1]}} $\\pm$ {agg.loc[idx, (col, 'std')]:{FLOAT_FMT[2:-1]}}"
            for col in metric_cols
        }
    return pd.DataFrame(rows).T


def module_by_shipgroup(long, df_hp, module_col, metric="ade", region="kiel"):
    """Mean metric per (ship_group, module setting), averaged over models & horizon."""
    df = long[long["region"] == region]
    # one scalar per model & ship_group (scalar metrics repeat across steps)
    per_model = df.groupby(["dir_name", "ship_group"])[metric].mean().reset_index()
    # attach the module setting, normalize "off"
    setting = df_hp[module_col].replace({None: "off", "null": "off", "None": "off"})
    setting = setting.where(setting.notna(), "off")
    per_model[module_col] = per_model["dir_name"].map(setting)
    return per_model




In [ ]:
# ── Config ───────────────────────────────────────────────────────────────────
DISPLAY_SCALAR = ["ade", "k_ade"]# "collision_ratio"
#FDE_STEPS      = [6,18, 30]
FDE_STEPS = []
SHOW_STD       = True          # ← toggle ± std in all cells
FLOAT_FMT      = "{:.4f}"

MODULE_ABBREV = {
    "nereus_modules.social": "Social",
    "nereus_modules.map":    "Map",
    "nereus_modules.prior":  "Prior",
}
METRIC_MAP = {"ade":"ADE", "k_ade":"minADE", "fde":"FDE", "k_fde":"minFDE"}


# ── Render LaTeX ─────────────────────────────────────────────────────────────
for module_col in MODULE_COLS:
    title = MODULE_ABBREV[module_col]
    tbl = build_table(df_eval_scalar, df_eval_step,fde_steps=FDE_STEPS, group_col=module_col,show_std=SHOW_STD)
    tbl.rename(columns=METRIC_MAP, inplace=True)

    tbl.index.name = title

    # String cells need escape=False so \pm is not escaped
    latex = tbl.to_latex(
        float_format="%.4f",
        caption=f"Ablation: effect of the {title} module (mean $\\pm$ std over other module combinations).",
        label=f"tab:ablation_{module_col.split('.')[-1]}",
        column_format="l" + "r" * len(tbl.columns),
        escape=False)

    #display(tbl)
    print(latex)
    print()

In [ ]:

df_eval_scalar.groupby(['nereus_modules.prior','nereus_modules.social', 'nereus_modules.map']).agg("mean")[["ade","k_ade"]]

In [ ]:
df_eval_scalar.groupby(['nereus_modules.map','nereus_modules.prior','nereus_modules.social']).agg("mean")[["ade","k_ade"]]

In [ ]:
df_eval_scalar.groupby(MODULE_COLS).agg("mean")[["ade","k_ade"]]#.loc["None"] - df_eval_scalar.groupby(MODULE_COLS).agg("mean")[["ade","k_ade"]].loc["pool"]

In [ ]:
eval_long[(eval_long["step"] == 1) & (eval_long["ship_group"] == "all")]

In [ ]:
import plotly.express as px

x="nereus_modules.prior"#"nereus_modules.social"
y="nereus_modules.map"
z="ade"
color = "nereus_modules.social" #ade

fig = px.scatter_3d(
    df_eval_scalar,
    x=x,
    y=y,
    z=z,
    color=color,
    color_continuous_scale="Viridis",
)
fig.update_traces(marker=dict(size=4))
fig.update_layout(
    scene=dict(
        xaxis_title=x,
        yaxis_title=y,
        zaxis_title=z,
    )
)
fig.show()


In [ ]:
df_hp.T["version_15"]



In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

SHIP_GROUPS = ["all", "cargo", "passenger", "sailing", "other"]
PLOT_METRIC = "ade"
#PLOT_METRIC = "k_ade"
METRIC_LABELS = {
    "ade": "ADE [m]",
    "k_ade": "minADE [m]",
    "fde": "FDE [m]",
    "k_fde": "minFDE [m]",
}



# ── Collect data + one fixed color per setting across all subplots ──────────
per_module_data = {m: module_by_shipgroup(eval_long, df_hp, m, metric=PLOT_METRIC)
                   for m in MODULE_COLS}
all_settings = sorted(
    {s for m, pm in per_module_data.items() for s in pm[m].unique()},
    key=lambda x: (x != "off", x),
)
setting_color = {s: PAPER_COLORS[i % len(PAPER_COLORS)] for i, s in enumerate(all_settings)}

# ── Subplots ─────────────────────────────────────────────────────────────────
fig = make_subplots(
    rows=1, cols=3, shared_yaxes=True,
    subplot_titles=[MODULE_TITLES[m] for m in MODULE_COLS],
    horizontal_spacing=0.04,
)

for i, module_col in enumerate(MODULE_COLS, start=1):
    pm = per_module_data[module_col]
    settings = sorted(pm[module_col].unique(), key=lambda x: (x != "off", x))

    legend_key = "legend" if i == 1 else f"legend{i}"
    for s in settings:
        sub = pm[pm[module_col] == s]
        fig.add_trace(
            go.Box(
                x=sub["ship_group"], y=sub[PLOT_METRIC],
                name=s,
                marker_color=setting_color[s],
                offsetgroup=s,
                legend=legend_key,                      # per-subplot legend
                legendgroup=f"{module_col}/{s}",
            ),
            row=1, col=i,
        )
    fig.update_xaxes(categoryorder="array", categoryarray=SHIP_GROUPS, row=1, col=i)

# ── One horizontal legend centered under each subplot ───────────────────────
for i, module_col in enumerate(MODULE_COLS, start=1):
    legend_key = "legend" if i == 1 else f"legend{i}"
    axis_key = "xaxis" if i == 1 else f"xaxis{i}"
    x0, x1 = fig.layout[axis_key].domain
    fig.update_layout({legend_key: dict(
        orientation="h",
        x=(x0 + x1) / 2, xanchor="center",
        y=-0.15, yanchor="top",
    )})

fig.update_layout(
    boxmode="group",
    width=1100, height=420,
    margin=dict(t=40, b=80),
)
#fig.update_yaxes(title_text=PLOT_METRIC, row=1, col=1)
fig.update_yaxes(title_text=METRIC_LABELS.get(PLOT_METRIC, PLOT_METRIC), row=1, col=1)

fig.show()
fig.write_image(f"figures/shipgroup_modules_{PLOT_METRIC}.pdf")


In [ ]:
PLOT_METRIC_STEP = "fde"   # "fde" or "k_fde"
SHOW_BANDS = False          # ±1 std band over runs with the same setting

fig = make_subplots(
    rows=1, cols=3, shared_yaxes=True,
    subplot_titles=[MODULE_TITLES[m] for m in MODULE_COLS],
    horizontal_spacing=0.04,
)

df = eval_long[(eval_long["ship_group"] == "all") & (eval_long["region"] == "kiel")]

for i, module_col in enumerate(MODULE_COLS, start=1):
    sub = df.copy()
    sub["setting"] = _setting_label(sub["dir_name"].map(df_hp[module_col]))
    stats = (sub.groupby(["setting", "minute"])[PLOT_METRIC_STEP]
                .agg(mean="mean", std="std").reset_index().sort_values("minute"))

    legend_key = "legend" if i == 1 else f"legend{i}"
    settings = sorted(stats["setting"].unique(), key=lambda x: (x != "off", x))
    for setting in settings:
        sdf = stats[stats["setting"] == setting]
        color = _setting_color(setting)
        x, y, e = sdf["minute"].to_numpy(), sdf["mean"].to_numpy(), sdf["std"].fillna(0).to_numpy()
        if SHOW_BANDS:
            fig.add_trace(go.Scatter(
                x=np.concatenate([x, x[::-1]]), y=np.concatenate([y + e, (y - e)[::-1]]),
                fill="toself", fillcolor=_rgba(color, 0.15), line=dict(width=0),
                hoverinfo="skip", showlegend=False,
                legendgroup=f"{module_col}/{setting}",
            ), row=1, col=i)
        fig.add_trace(go.Scatter(
            x=x, y=y, mode="lines", name=setting,
            line=dict(color=color, width=2.5),
            legend=legend_key, legendgroup=f"{module_col}/{setting}",
        ), row=1, col=i)
    fig.update_xaxes(title_text="Prediction horizon [min]", row=1, col=i)

# One horizontal legend centered under each subplot
for i, module_col in enumerate(MODULE_COLS, start=1):
    legend_key = "legend" if i == 1 else f"legend{i}"
    axis_key = "xaxis" if i == 1 else f"xaxis{i}"
    x0, x1 = fig.layout[axis_key].domain
    fig.update_layout({legend_key: dict(
        orientation="h",
        x=(x0 + x1) / 2, xanchor="center",
        y=-0.22, yanchor="top",
    )})

fig.update_layout(
    template="simple_white",
    width=1100, height=420, font=dict(size=15),
    margin=dict(t=40, b=95),
)
fig.update_yaxes(title_text=METRIC_LABELS.get(PLOT_METRIC_STEP, PLOT_METRIC_STEP), row=1, col=1)
fig.show()
fig.write_image(f"figures/ablation_modules_{PLOT_METRIC_STEP}.pdf")


In [ ]:
PLOT_METRIC_STEP = "k_fde"   # "fde" or "k_fde"
SHOW_BANDS = False
SHIP_GROUPS = ["all", "cargo", "passenger", "sailing", "other"]

n_rows = len(SHIP_GROUPS)
fig = make_subplots(
    rows=n_rows, cols=3,
    shared_xaxes=True, shared_yaxes=True,   # y shared within each row
    subplot_titles=[MODULE_TITLES[m] for m in MODULE_COLS] + [""] * (3 * (n_rows - 1)),
    horizontal_spacing=0.04, vertical_spacing=0.03,
)

df_all = eval_long[eval_long["region"] == "kiel"]

for r, ship_group in enumerate(SHIP_GROUPS, start=1):
    df = df_all[df_all["ship_group"] == ship_group]
    for c, module_col in enumerate(MODULE_COLS, start=1):
        sub = df.copy()
        sub["setting"] = _setting_label(sub["dir_name"].map(df_hp[module_col]))
        stats = (sub.groupby(["setting", "minute"])[PLOT_METRIC_STEP]
                    .agg(mean="mean", std="std").reset_index().sort_values("minute"))

        legend_key = "legend" if c == 1 else f"legend{c}"
        settings = sorted(stats["setting"].unique(), key=lambda x: (x != "off", x))
        for setting in settings:
            sdf = stats[stats["setting"] == setting]
            color = _setting_color(setting)
            x, y, e = sdf["minute"].to_numpy(), sdf["mean"].to_numpy(), sdf["std"].fillna(0).to_numpy()
            if SHOW_BANDS:
                fig.add_trace(go.Scatter(
                    x=np.concatenate([x, x[::-1]]), y=np.concatenate([y + e, (y - e)[::-1]]),
                    fill="toself", fillcolor=_rgba(color, 0.15), line=dict(width=0),
                    hoverinfo="skip", showlegend=False,
                    legendgroup=f"{module_col}/{setting}",
                ), row=r, col=c)
            fig.add_trace(go.Scatter(
                x=x, y=y, mode="lines", name=setting,
                line=dict(color=color, width=2.5),
                legend=legend_key, legendgroup=f"{module_col}/{setting}",
                showlegend=(r == 1),          # one legend entry per column
            ), row=r, col=c)

    # row label: ship group + metric on the left axis
    fig.update_yaxes(
        title_text=f"<b>{ship_group}</b><br>{METRIC_LABELS.get(PLOT_METRIC_STEP, PLOT_METRIC_STEP)}",
        row=r, col=1,
    )

for c in range(1, 4):
    fig.update_xaxes(title_text="Prediction horizon [min]", row=n_rows, col=c)

# One horizontal legend centered under each column
for c, module_col in enumerate(MODULE_COLS, start=1):
    legend_key = "legend" if c == 1 else f"legend{c}"
    axis_key = "xaxis" if c == 1 else f"xaxis{c}"
    x0, x1 = fig.layout[axis_key].domain
    fig.update_layout({legend_key: dict(
        orientation="h",
        x=(x0 + x1) / 2, xanchor="center",
        y=-0.06, yanchor="top",
    )})

fig.update_layout(
    template="simple_white",
    width=1100, height=250 * n_rows, font=dict(size=13),
    margin=dict(t=40, b=90, l=80),
)
fig.show()
fig.write_image(f"figures/ablation_modules_shipgroups_{PLOT_METRIC_STEP}.pdf")


In [ ]:
df_eval_scalar.sort_values("ade")#.head(10)[["ade", "k_ade", "collision_ratio"] + MODULE_COLS]

In [ ]:
# # One figure per module: how each module shifts the FDE-vs-horizon curve.
# # Saved to figures/ as PDF (needs `kaleido`); falls back to on-screen only.
# Path("figures").mkdir(exist_ok=True)

# for module in MODULE_COLS:
#     fig = plot_module_effect(eval_long, df_hp, module, metric="fde", region="kiel")
#     fig.show()
#     try:
#         fig.write_image(f"figures/ablation_{module.split('.')[-1]}_fde.pdf")
#     except Exception as e:  # kaleido not installed
#         print("PDF export skipped (pip install kaleido):", e)
#         #fig.write_image(f"figures/ablation_{module.split('.')[-1]}_fde.pdf")
# #fig.write_image(f"figures/ablation_{module.split('.')[-1]}_fde.pdf")




In [ ]:
# Same plots straight from the old tbparse df_step_metrics (all 30 runs already logged).
# region=None because the training-time series logs have no region column.
# step_long = step_metrics_to_long(df_step_metrics, split="test")

# for module in MODULE_COLS:
#     plot_module_effect(step_long, df_hp, module, metric="fde", region=None,show_bands=False).show()


## GRU

In [ ]:
log_dir_gru = "lightning_logs/baselines/"

reader_gru = SummaryReader(log_dir_gru,extra_columns={"dir_name"})
df_metrics_gru = reader_gru.scalars
df_loss_gru = load_metrics(reader_gru)
df_hp_gru = load_hp(reader_gru)
df_step_metrics_gru = load_step_metrics(reader_gru)

In [ ]:
df_loss_gru["train_loss"].plot()

In [ ]:

pivot = df_metrics_gru[df_metrics_gru["tag"] == "val_metric"].pivot(columns="dir_name", values="value", index="step")
fig = go.Figure()
for col in pivot.columns:
    s = pivot[col].dropna()
    fig.add_trace(go.Scatter(x=s.index, y=s.values, mode="lines", name=col))
fig.show()


#df_loss_gru["val_metric"].plot()

In [ ]:
df_metrics_gru[df_metrics_gru["tag"] == "ade/test"]#.pivot(columns="dir_name", values="value", index="step").plot()

In [ ]:
df_metrics[df_metrics["tag"] == "ade/test"]#.pivot(columns="dir_name", values="value", index="step").plot()

In [ ]:
df_step_metrics_gru["step_de/val"]

In [ ]:
# Load the new metrics and plug them into the existing analyzer / hp table.
eval_long_gru = load_eval_csvs(log_dir_gru, source="fh")
df_eval_step_gru = eval_step_metrics(eval_long_gru, region="kiel", ship_group="all")
#df_eval_scalar_gru = eval_scalar_metrics(eval_long_gru, region="kiel", ship_group="all")

print("models evaluated:", eval_long_gru["dir_name"].nunique(), "|", sorted(eval_long_gru["dir_name"].unique()))

# Scalars joined with the module hyperparameters, ready for tables/plots.
#df_eval_scalar_gru = df_eval_scalar_gru.join(df_hp_gru[MODULE_COLS])


In [ ]:
eval_long_gru

In [ ]:
df_eval_step_gru

In [ ]:
df_metrics_gru[df_metrics_gru["tag"] == "val_metric"]


## IS-STGCNN

In [ ]:
#log_dir = "lightning_logs/nereus"
log_dir = "lightning_logs/isstgcnn/"

reader = SummaryReader(log_dir,extra_columns={"dir_name"})
df_metrics = reader.scalars
df_loss = load_metrics(reader)
df_hp = load_hp(reader)
df_step_metrics = load_step_metrics(reader)


#SETTING_COLORS = build_setting_colors(df_hp)


In [ ]:
df_metrics

In [ ]:
df_step_metrics["step_de/test"]

### ?

In [ ]:
log_dir_desire = "lightning_logs/desire/"

reader_desire = SummaryReader(log_dir_desire,extra_columns={"dir_name"})
df_metrics_desire = reader_desire.scalars

In [ ]:
df_metrics_desire[df_metrics_desire["tag"] =="epoch"]

In [ ]:
df_metrics_desire[df_metrics_desire["tag"] =="val_metric"]["value"].plot()
